# Installation:


conda create -n pysparklesson python=3.9  
conda activate pysparklesson  
conda install -c conda-forge pyspark  
pip install jupyter  
jupyter notebook  

In [1]:
from pyspark.sql import SparkSession,Row, functions as sql_f
from pyspark import SparkContext, SQLContext,SparkConf

# 
from pyspark.sql.functions import pandas_udf, PandasUDFType

# schema 
from pyspark.sql.types import *

import pandas as pd
import numpy as np

In [2]:
# define spark context (boilerplate)

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

In [40]:
checkpoint_dir = "/tmp/spark-checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

In [42]:
spark = (SparkSession.builder
    .master("local[*]")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate())

sc = spark.sparkContext
sc.setCheckpointDir("/tmp/spark-checkpoints")

In [41]:
### conf = SparkConf().setMaster("local[*]").set("spark.sql.execution.arrow.pyspark.enabled", "true")
sc = SparkContext(conf=conf)
sc.setCheckpointDir("/tmp/spark-checkpoints")
spark = SparkSession(sc)

ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=pyspark-shell, master=local[*]) created by __init__ at /tmp/ipykernel_4942/3920356536.py:2 

# Load csv

In [4]:
train = spark.read.csv(path="data/train.csv",
                      header=True,
                      inferSchema=True)

In [5]:
train.show(n=10)

+-------+----------+------+-----+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|User_ID|Product_ID|Gender|  Age|Occupation|City_Category|Stay_In_Current_City_Years|Marital_Status|Product_Category_1|Product_Category_2|Product_Category_3|Purchase|
+-------+----------+------+-----+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|1000001| P00069042|     F| 0-17|        10|            A|                         2|             0|                 3|              NULL|              NULL|    8370|
|1000001| P00248942|     F| 0-17|        10|            A|                         2|             0|                 1|                 6|                14|   15200|
|1000001| P00087842|     F| 0-17|        10|            A|                         2|             0|                12|              NULL|              NULL|    1422

In [8]:
train.printSchema()

root
 |-- User_ID: integer (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- City_Category: string (nullable = true)
 |-- Stay_In_Current_City_Years: string (nullable = true)
 |-- Marital_Status: integer (nullable = true)
 |-- Product_Category_1: integer (nullable = true)
 |-- Product_Category_2: integer (nullable = true)
 |-- Product_Category_3: integer (nullable = true)
 |-- Purchase: integer (nullable = true)



# Shape

In [6]:
train.count()

550068

In [7]:
len(train.columns)

12

# Filtering

In [8]:
train_filtered = train.filter(train.Purchase > 100)

In [9]:
train_filtered_2 = train.where(train.Purchase > 100)

In [10]:
assert train_filtered_2.count() == train_filtered.count()

# Select columns, sum, rename

In [11]:
train_filtered = train_filtered.select("User_ID",
                                       "Purchase"
                                      )

In [12]:
train_filtered = train.where(train.Purchase > 100).select("Purchase")

In [13]:
train_filtered = train.where(train.Purchase > 100).\
                 select(sql_f.sum("Purchase").\
                        alias("total_purchase_value"))

In [14]:
train_filtered.show()

+--------------------+
|total_purchase_value|
+--------------------+
|          5095753364|
+--------------------+



In [15]:
total_distinct_users = train.where(train.Purchase > 100).\
                       select(sql_f.countDistinct("User_ID").alias("total_unique_users"))

In [16]:
total_distinct_users.collect()[0]["total_unique_users"]

5891

In [38]:
train.where(train.Purchase > 100).\
                       select("User_ID").distinct().count()

5891

# Group by

In [17]:
total_product_category_1_purchase = train.\
                                    groupBy("Product_Category_1").\
                                    agg(sql_f.sum("Purchase").alias("total_purchase"),
                                       sql_f.countDistinct("Purchase").alias("total_unique_customers"))

In [18]:
total_product_category_1_purchase.show()

+------------------+--------------+----------------------+
|Product_Category_1|total_purchase|total_unique_customers|
+------------------+--------------+----------------------+
|                12|       5331844|                   334|
|                 1|    1910013754|                  3795|
|                13|       4008601|                   189|
|                 6|     324150302|                  3417|
|                16|     145120612|                  3288|
|                 3|     204084713|                  2528|
|                20|        944727|                   120|
|                 5|     941835229|                  1715|
|                19|         59378|                    15|
|                15|      92969042|                  3073|
|                 9|       6370324|                   394|
|                17|       5878699|                   487|
|                 4|      27380488|                   685|
|                 8|     854318799|                  194

# Dropping duplicates

In [19]:
train.select("Age").dropDuplicates().\
show()

+-----+
|  Age|
+-----+
|18-25|
|26-35|
| 0-17|
|46-50|
|51-55|
|36-45|
|  55+|
+-----+



# Joins

In [20]:
dummy1 = pd.DataFrame()
dummy1["key"] = np.arange(1000)
dummy1["values"] = np.random.randn(1000)

In [21]:
dummy1

,key,values
0,0,1.274659
1,1,-0.562478
2,2,0.389159
3,3,0.566423
4,4,-0.113879
...,...,...
995,995,-0.730126
996,996,0.612984
997,997,-1.467938
998,998,-1.335897


In [22]:
dummy2 = pd.DataFrame()
dummy2['key'] = np.random.choice(np.arange(1000), size = 500, replace = False)

In [23]:
dummy2

,key
0,69
1,495
2,736
3,943
4,726
...,...
495,262
496,318
497,801
498,445


In [24]:
dummy1_spark = spark.createDataFrame(dummy1)
dummy2_spark = spark.createDataFrame(dummy2)

In [25]:
dummy1_spark.show()

+---+--------------------+
|key|              values|
+---+--------------------+
|  0|   1.274658850036408|
|  1| -0.5624775616789557|
|  2| 0.38915878491632194|
|  3|  0.5664231343828531|
|  4|-0.11387944778313333|
|  5| -0.6968489826255032|
|  6| -0.9534006221243919|
|  7|  1.1519381942182383|
|  8| 0.21703943677419324|
|  9|  0.4040062023684107|
| 10|  0.7384051140198239|
| 11|  1.3193824673135346|
| 12|  0.9893612455719688|
| 13|  1.2501261011975253|
| 14|   0.519712225567317|
| 15| 0.01338532123168469|
| 16|  0.3712773250445561|
| 17|   0.784399864305263|
| 18|   1.273402731447641|
| 19| -0.8005294381803668|
+---+--------------------+
only showing top 20 rows


In [26]:
join_result = dummy2_spark.join(dummy1_spark,
                               on = ["key"],
                               how= "inner")

In [27]:
join_result.show()

+---+--------------------+
|key|              values|
+---+--------------------+
|  0|   1.274658850036408|
|  3|  0.5664231343828531|
|  5| -0.6968489826255032|
|  7|  1.1519381942182383|
| 10|  0.7384051140198239|
| 11|  1.3193824673135346|
| 13|  1.2501261011975253|
| 14|   0.519712225567317|
| 20|  0.5659046125967249|
| 21| -0.9844331887678608|
| 23|  0.6923365147204485|
| 26| 0.07005545163587817|
| 27|  1.1751619504497253|
| 28|  0.3160045203978186|
| 30| 0.38441096833283916|
| 37|  0.1179014560996998|
| 38|  1.4959326930527903|
| 39|-0.47650880605582907|
| 43|  1.0525470927324914|
| 45| -1.6677083167443185|
+---+--------------------+
only showing top 20 rows


# Drop na

In [28]:
train.dropna().count()

166821

# Fill na

In [29]:
train_filled_na = train.select("Purchase").fillna(-1)

In [30]:
train_filled_na.filter(train_filled_na.Purchase == -1).show()

+--------+
|Purchase|
+--------+
+--------+



# Sample

In [31]:
train.sample(withReplacement=None,fraction=0.1).count()

55061

# Adding a column

In [32]:
train = train.withColumn("new_column", train.Purchase / 2.0)

In [33]:
train.show()

+-------+----------+------+-----+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+----------+
|User_ID|Product_ID|Gender|  Age|Occupation|City_Category|Stay_In_Current_City_Years|Marital_Status|Product_Category_1|Product_Category_2|Product_Category_3|Purchase|new_column|
+-------+----------+------+-----+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+----------+
|1000001| P00069042|     F| 0-17|        10|            A|                         2|             0|                 3|              NULL|              NULL|    8370|    4185.0|
|1000001| P00248942|     F| 0-17|        10|            A|                         2|             0|                 1|                 6|                14|   15200|    7600.0|
|1000001| P00087842|     F| 0-17|        10|            A|                         2|             0|          

# UDF

https://www.databricks.com/blog/2020/05/20/new-pandas-udfs-and-python-type-hints-in-the-upcoming-release-of-apache-spark-3-0.html?utm_source=hootsuite&utm_medium=&utm_term=&utm_content=&utm_campaign=

In [34]:
# mean group by agg
train.groupBy('User_ID').agg(sql_f.mean(train.Purchase)).show(n=10)

+-------+------------------+
|User_ID|     avg(Purchase)|
+-------+------------------+
|1000149| 11302.68263473054|
|1000190| 9612.111111111111|
|1000636| 12100.44642857143|
|1001043|11461.695652173914|
|1001129|10781.338461538462|
|1001139|10564.731481481482|
|1001601|13003.398058252427|
|1002605| 10818.85436893204|
|1003031|            8917.2|
|1003373|10867.411764705883|
+-------+------------------+
only showing top 10 rows


In [35]:
"""
@pandas_udf(df.schema, PandasUDFType.GROUPED_MAP)
def subtract_mean(pdf):
    v = pdf.v
    return pdf.assign(v=v - v.mean())

"""

schema = StructType([StructField("User_ID", IntegerType(), True),
                    StructField("output", DoubleType(), True)])

@pandas_udf(schema, PandasUDFType.GROUPED_MAP)
def just_some_custom_mean(df):
    """
    Parameters
    ----------
    df: pd.DataFrame
    """
    key = df["User_ID"].unique()[0]
    output = df["Purchase"].mean()
    return pd.DataFrame({"User_ID":[key],
                        "output":[output]})

In [36]:
# train_pandas.groupby(["User_ID"]).apply(just_some_custom_mean) - not distributed
result = train.groupBy("User_ID").apply(just_some_custom_mean)

/home/liuyx/miniconda3/envs/pysparklesson/lib/python3.9/site-packages/pyspark/sql/pandas/group_ops.py:122: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(


In [37]:
result.show()

[Stage 47:>                                                         (0 + 1) / 1]

+-------+------------------+
|User_ID|            output|
+-------+------------------+
|1000049|12206.066666666668|
|1000055| 9594.285714285714|
|1000060| 7778.583333333333|
|1000062| 8913.539007092199|
|1000083|  9999.22641509434|
|1000084| 12596.59090909091|
|1000088| 9980.242424242424|
|1000089| 8019.714285714285|
|1000092| 7057.910313901345|
|1000101|17246.439393939392|
|1000123|  9196.28409090909|
|1000127|          11712.61|
|1000144|           10638.5|
|1000149| 11302.68263473054|
|1000162|12479.236363636364|
|1000164| 8655.166666666666|
|1000174|13631.732142857143|
|1000190| 9612.111111111111|
|1000195| 9704.552462526766|
|1000203| 5985.953846153846|
+-------+------------------+
only showing top 20 rows


# checkpoint/cache

In [43]:
train.checkpoint() # save and execute all actions 

DataFrame[User_ID: int, Product_ID: string, Gender: string, Age: string, Occupation: int, City_Category: string, Stay_In_Current_City_Years: string, Marital_Status: int, Product_Category_1: int, Product_Category_2: int, Product_Category_3: int, Purchase: int, new_column: double]